In [1]:
import os
import base64
import pymupdf as fitz
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage

In [2]:
# ==========================================
# 1. SCHEMAS (Pydantic Models)
# ==========================================

class PageBlocks(BaseModel):
    blocks: List[str] = Field(
        description="List of raw, faithfully transcribed text, formulas, or code blocks from the page."
    )

class TopicChunk(BaseModel):
    main_topic: str = Field(
        description="Top-level concept (e.g., 'Divide and Conquer Method', 'Merge Sort', 'Master Method', 'Knapsack Problem')"
    )
    sub_topic: Optional[str] = Field(
        None, 
        description="Specific sub-section (e.g., 'General Recurrence Derivation', 'Iterative Algorithm', 'Execution Trace Example')"
    )
    content: str = Field(
        description="Complete, merged text, formulas, and steps belonging to this topic/subtopic."
    )

class DocumentStructured(BaseModel):
    chunks: List[TopicChunk] = Field(
        description="List of all cohesive topic chunks extracted from the entire document."
    )

In [3]:
# ==========================================
# 2. MODEL INITIALIZATIONS
# ==========================================

# Vision OCR Model (Pixtral 12B)
vision_model = ChatMistralAI(
    model="pixtral-12b-2409",
    temperature=0,
    api_key=os.getenv("MISTRAL_API_KEY_1")
)
transcribe_llm = vision_model.with_structured_output(schema=PageBlocks)

# Text Reasoning Model for Structuring (Mistral Large / Medium 3.5)
reasoning_model = ChatMistralAI(
    model="mistral-medium-3-5",  # or "mistral-medium-2312" / "mistral-small-latest"
    temperature=0,
    api_key=os.getenv("MISTRAL_API_KEY_1")
)
structure_llm = reasoning_model.with_structured_output(schema=DocumentStructured)


In [4]:
# ==========================================
# 3. PROMPTS & HELPERS
# ==========================================

TRANSCRIBE_PROMPT = """Transcribe all handwritten and printed text on this page faithfully, in natural reading order.
- Break it into natural blocks (headings, paragraphs, pseudo-code/algorithms, formulas, worked derivation steps).
- Wrap mathematical equations in LaTeX notation ($...$ or $$...$$).
- Transcribe code/algorithms with proper indentation.
- Mark completely unreadable words as [illegible].
- Skip printed margins, running headers/footers, and page numbers.

OUTPUT FORMAT:
Return ONLY a valid JSON object matching this schema:
{
  "blocks": ["block 1 text...", "block 2 text...", "..."]
}
"""

GROUP_PROMPT = """You are an expert technical text organizer for computer science academic notes.
You are given OCR-extracted text blocks from a multi-page document in sequential order.

Your task is to organize and merge these blocks into coherent, topic-based chunks.

Rules:
1. Merge multi-page continuations: If an algorithm, worked example, recurrence derivation, or trace splits across pages, combine it into ONE complete chunk.
2. Keep full execution traces intact: Do not split step-by-step array partition trees, sort passes, or recurrence trees into micro-chunks. Keep the entire example whole.
3. Content Fidelity: Preserve all original mathematical steps, formulas, and code. Do not summarize or omit steps.
4. Categorize correctly: Provide an accurate `main_topic` and `sub_topic` for each chunk.
"""

In [5]:
def convert_page_to_image(pdf_path: str, page_num: int = 0, dpi: int = 140) -> str:
    """Renders a PDF page to base64 JPEG to avoid context length overflow."""
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    jpeg_bytes = pix.tobytes("jpeg", jpg_quality=80)
    return base64.b64encode(jpeg_bytes).decode("utf-8")

def build_transcribe_msg(img_b64: str):
    return [
        HumanMessage(content=[
            {"type": "text", "text": TRANSCRIBE_PROMPT},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
        ])
    ]

In [6]:
# ==========================================
# 4. EXECUTION PIPELINE
# ==========================================

pdf_path = r'C:\Voice Agent\testing\docs\DM - 2.pdf'
doc = fitz.open(pdf_path)

# Step 1: Transcribe each page into discrete blocks
all_blocks = []
print(f"Starting OCR transcription across {len(doc)} pages...")

for i in range(len(doc)):
    img_b64 = convert_page_to_image(pdf_path, page_num=i, dpi=140)
    response = transcribe_llm.invoke(build_transcribe_msg(img_b64))
    if response is not None:
        all_blocks.extend(response.blocks)
        print(f"Transcribed page {i + 1}/{len(doc)}")

# Step 2: Combine blocks and semantically structure them globally
print("\nStructuring transcribed blocks into coherent topic chunks...")
full_transcription = "\n\n---BLOCK---\n\n".join(all_blocks)

result = structure_llm.invoke([
    HumanMessage(content=f"{GROUP_PROMPT}\n\nDOCUMENT BLOCKS:\n{full_transcription}")
])

# Step 3: Print and inspect the chunks
print("\n" + "=" * 80 + "\nSTRUCTURED CHUNKS OUTPUT:\n" + "=" * 80)
for idx, chunk in enumerate(result.chunks, 1):
    print(f"\n[Chunk {idx}]")
    print(f"MAIN TOPIC : {chunk.main_topic}")
    print(f"SUB TOPIC  : {chunk.sub_topic}")
    print("CONTENT    :\n" + chunk.content)
    print("-" * 80)

Starting OCR transcription across 17 pages...
Transcribed page 1/17
Transcribed page 2/17
Transcribed page 3/17
Transcribed page 4/17
Transcribed page 5/17
Transcribed page 6/17
Transcribed page 7/17
Transcribed page 8/17
Transcribed page 9/17
Transcribed page 10/17
Transcribed page 11/17
Transcribed page 12/17
Transcribed page 13/17
Transcribed page 14/17
Transcribed page 16/17
Transcribed page 17/17

Structuring transcribed blocks into coherent topic chunks...

STRUCTURED CHUNKS OUTPUT:

[Chunk 1]
MAIN TOPIC : Introduction to Data Mining
SUB TOPIC  : Overview and Motivation
CONTENT    :
UNIT –I: Introduction: Why Data Mining? What Is Data Mining? 

1.1. Why Data Mining?
- The major reason that data mining has attracted a great deal of attention in the information industry in recent years is due to the wide availability of huge amounts of data and the need for turning such data into useful information and knowledge.
- The information and knowledge gained can be used for applications r